# Data Cleaning and Preprocessing

This notebook performs data validation, cleaning, and preprocessing of the raw Delhi weather dataset collected from the Open-Meteo API.

The raw dataset is preserved in `data/raw/` and will not be modified directly.

### Objectives

- Load the raw weather dataset
- Validate data types and timestamps
- Check for missing values and duplicates
- Validate environmental variable ranges
- Check timestamp continuity
- Handle only genuinely invalid records
- Create useful time-based features
- Save the cleaned dataset to `data/processed/`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
raw_path = Path("../data/raw/weather_delhi.csv")

df = pd.read_csv(raw_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (8760, 8)


,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_gusts_10m
0,2025-01-01 00:00:00,8.3,100,7.4,0.0,99,2.8,4.7
1,2025-01-01 01:00:00,8.0,100,7.0,0.0,100,2.9,5.4
2,2025-01-01 02:00:00,7.8,100,6.5,0.0,100,4.7,10.1
3,2025-01-01 03:00:00,8.0,99,6.5,0.0,100,5.8,12.6
4,2025-01-01 04:00:00,7.7,98,6.1,0.0,100,6.3,13.0


In [3]:
cleaned_df = df.copy()

In [4]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   time                  8760 non-null   str    
 1   temperature_2m        8760 non-null   float64
 2   relative_humidity_2m  8760 non-null   int64  
 3   apparent_temperature  8760 non-null   float64
 4   precipitation         8760 non-null   float64
 5   cloud_cover           8760 non-null   int64  
 6   wind_speed_10m        8760 non-null   float64
 7   wind_gusts_10m        8760 non-null   float64
dtypes: float64(5), int64(2), str(1)
memory usage: 547.6 KB


In [5]:
cleaned_df["time"] = pd.to_datetime(
    cleaned_df["time"],
    errors="coerce"
)

cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  8760 non-null   datetime64[us]
 1   temperature_2m        8760 non-null   float64       
 2   relative_humidity_2m  8760 non-null   int64         
 3   apparent_temperature  8760 non-null   float64       
 4   precipitation         8760 non-null   float64       
 5   cloud_cover           8760 non-null   int64         
 6   wind_speed_10m        8760 non-null   float64       
 7   wind_gusts_10m        8760 non-null   float64       
dtypes: datetime64[us](1), float64(5), int64(2)
memory usage: 547.6 KB


In [6]:
invalid_dates = cleaned_df["time"].isna().sum()

print("Invalid timestamps:", invalid_dates)

Invalid timestamps: 0


In [8]:
missing_values = cleaned_df.isnull().sum()

print(missing_values)

print("Total missing values:", cleaned_df.isnull().sum().sum())

time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
precipitation           0
cloud_cover             0
wind_speed_10m          0
wind_gusts_10m          0
dtype: int64
Total missing values: 0


In [9]:
duplicate_rows = cleaned_df.duplicated().sum()

print("Duplicate rows:", duplicate_rows)

Duplicate rows: 0


In [10]:
duplicate_timestamps = cleaned_df["time"].duplicated().sum()

print("Duplicate timestamps:", duplicate_timestamps)

Duplicate timestamps: 0


In [11]:
cleaned_df = cleaned_df.sort_values("time").reset_index(drop=True)

cleaned_df.head()

,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_gusts_10m
0,2025-01-01 00:00:00,8.3,100,7.4,0.0,99,2.8,4.7
1,2025-01-01 01:00:00,8.0,100,7.0,0.0,100,2.9,5.4
2,2025-01-01 02:00:00,7.8,100,6.5,0.0,100,4.7,10.1
3,2025-01-01 03:00:00,8.0,99,6.5,0.0,100,5.8,12.6
4,2025-01-01 04:00:00,7.7,98,6.1,0.0,100,6.3,13.0


In [12]:
print("Humidity range:")
print(
    cleaned_df["relative_humidity_2m"].min(),
    "to",
    cleaned_df["relative_humidity_2m"].max()
)

print("\nCloud cover range:")
print(
    cleaned_df["cloud_cover"].min(),
    "to",
    cleaned_df["cloud_cover"].max()
)

print("\nPrecipitation range:")
print(
    cleaned_df["precipitation"].min(),
    "to",
    cleaned_df["precipitation"].max()
)

print("\nTemperature range:")
print(
    cleaned_df["temperature_2m"].min(),
    "to",
    cleaned_df["temperature_2m"].max()
)

Humidity range:
4 to 100

Cloud cover range:
0 to 100

Precipitation range:
0.0 to 12.0

Temperature range:
7.7 to 43.7


In [13]:
invalid_humidity = cleaned_df[
    (cleaned_df["relative_humidity_2m"] < 0) |
    (cleaned_df["relative_humidity_2m"] > 100)
]

print("Invalid humidity records:", len(invalid_humidity))

Invalid humidity records: 0


In [14]:
invalid_cloud = cleaned_df[
    (cleaned_df["cloud_cover"] < 0) |
    (cleaned_df["cloud_cover"] > 100)
]

print("Invalid cloud-cover records:", len(invalid_cloud))

Invalid cloud-cover records: 0


In [15]:
invalid_precipitation = cleaned_df[
    cleaned_df["precipitation"] < 0
]

print("Invalid precipitation records:", len(invalid_precipitation))

Invalid precipitation records: 0


In [16]:
invalid_wind = cleaned_df[
    (cleaned_df["wind_speed_10m"] < 0) |
    (cleaned_df["wind_gusts_10m"] < 0)
]

print("Invalid wind records:", len(invalid_wind))

Invalid wind records: 0


In [17]:
time_difference = cleaned_df["time"].diff()

print(time_difference.value_counts().head())

time
0 days 01:00:00    8759
Name: count, dtype: int64


In [19]:
expected_interval = pd.Timedelta(hours=1)

time_difference = cleaned_df["time"].diff()

unexpected_intervals = time_difference.iloc[1:] != expected_interval

print(
    "Unexpected hourly intervals:",
    unexpected_intervals.sum()
)

Unexpected hourly intervals: 0


In [20]:
cleaned_df["year"] = cleaned_df["time"].dt.year
cleaned_df["month"] = cleaned_df["time"].dt.month
cleaned_df["day"] = cleaned_df["time"].dt.day
cleaned_df["hour"] = cleaned_df["time"].dt.hour
cleaned_df["day_of_year"] = cleaned_df["time"].dt.dayofyear

In [22]:
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"


cleaned_df["season"] = cleaned_df["month"].apply(assign_season)

cleaned_df["season"].value_counts()

season
Monsoon         2928
Summer          2208
Winter          2160
Post-Monsoon    1464
Name: count, dtype: int64

In [23]:
cleaned_df["rain_flag"] = (
    cleaned_df["precipitation"] > 0
).astype(int)

In [25]:
cleaned_df.head()

,time,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,cloud_cover,wind_speed_10m,wind_gusts_10m,year,month,day,hour,day_of_year,season,rain_flag
0,2025-01-01 00:00:00,8.3,100,7.4,0.0,99,2.8,4.7,2025,1,1,0,1,Winter,0
1,2025-01-01 01:00:00,8.0,100,7.0,0.0,100,2.9,5.4,2025,1,1,1,1,Winter,0
2,2025-01-01 02:00:00,7.8,100,6.5,0.0,100,4.7,10.1,2025,1,1,2,1,Winter,0
3,2025-01-01 03:00:00,8.0,99,6.5,0.0,100,5.8,12.6,2025,1,1,3,1,Winter,0
4,2025-01-01 04:00:00,7.7,98,6.1,0.0,100,6.3,13.0,2025,1,1,4,1,Winter,0


In [26]:
cleaned_df.shape

(8760, 15)

In [27]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   time                  8760 non-null   datetime64[us]
 1   temperature_2m        8760 non-null   float64       
 2   relative_humidity_2m  8760 non-null   int64         
 3   apparent_temperature  8760 non-null   float64       
 4   precipitation         8760 non-null   float64       
 5   cloud_cover           8760 non-null   int64         
 6   wind_speed_10m        8760 non-null   float64       
 7   wind_gusts_10m        8760 non-null   float64       
 8   year                  8760 non-null   int32         
 9   month                 8760 non-null   int32         
 10  day                   8760 non-null   int32         
 11  hour                  8760 non-null   int32         
 12  day_of_year           8760 non-null   int32         
 13  season                8760 no

In [28]:
print("Final dataset shape:", cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate rows:")
print(cleaned_df.duplicated().sum())

print("\nDuplicate timestamps:")
print(cleaned_df["time"].duplicated().sum())

Final dataset shape: (8760, 15)

Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
precipitation           0
cloud_cover             0
wind_speed_10m          0
wind_gusts_10m          0
year                    0
month                   0
day                     0
hour                    0
day_of_year             0
season                  0
rain_flag               0
dtype: int64

Duplicate rows:
0

Duplicate timestamps:
0


In [29]:
processed_path = Path("../data/processed/weather_delhi_cleaned.csv")

processed_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

cleaned_df.to_csv(
    processed_path,
    index=False
)

print(f"Processed dataset saved to: {processed_path}")

Processed dataset saved to: ..\data\processed\weather_delhi_cleaned.csv
